# All Steps of Piepline

In [2]:
import sys
import os

# Add the project root directory to sys.path
project_root = os.path.abspath("../../")  # Adjust the relative path as needed
if project_root not in sys.path:
    sys.path.append(project_root)

## Bulk download of CRS data

In [ ]:
# Import the necessary functions from src
from src.download_crs import get_full_crs_parquet_url, download_crs_parquet

# Get the full CRS parquet URL
file_url = get_full_crs_parquet_url()

# Download the CRS data
crs = download_crs_parquet(file_url)

In [ ]:
# Save raw CRS to feather 
crs.to_feather("../../data/raw/crs_raw.feather")

## A - Title Pattern Matching 

###  Keyword Detection

In [3]:
from src.text_processing import normalize_str, detect_language, lemmatize_batch, detect_keywords, detect_acronyms, process_keywords

#### Load keywords

In [ ]:
import pandas as pd
# Load keywords from Excel file
keywords_file_path = "../../data/keywords/review/keyword_review_modernization.xlsx"
stat_keywords = pd.read_excel(keywords_file_path, sheet_name="statistics")
gender_keywords = pd.read_excel(keywords_file_path, sheet_name="gender")
stat_acronyms = pd.read_excel(keywords_file_path, sheet_name="statistics acronyms")
gender_acronyms = pd.read_excel(keywords_file_path, sheet_name="gender acronyms")
stat_blacklist = pd.read_excel(keywords_file_path, sheet_name="stat blacklist")

# Drop id column
stat_keywords = stat_keywords.drop(columns=["id"])

# Process keywords
stat_keywords = process_keywords(stat_keywords)
gender_keywords = process_keywords(gender_keywords)
stat_blacklist = process_keywords(stat_blacklist)

#### Load CRS data

In [ ]:
# Apply process row function to the first 100 rows of the dataframe
#crs_test = crs.head(100000).copy()
crs_test = crs.copy()

# Reduce crs_test to only the columns we need
crs_test = crs_test[['project_title', 'short_description', 'long_description']]

# Only keep rows with project_title that are unique and keep the first occurrence
crs_test = crs_test.drop_duplicates(subset=['project_title'], keep='first')

#### Detect keywords

In [ ]:
# Process titles 
crs_test['normalized_title'] = crs_test['project_title'].apply(normalize_str)
crs_test['language'] = crs_test['normalized_title'].apply(detect_language)

In [ ]:
# Lemmatize in batches (for entire CRS dataset ~25min)
for lang in crs_test['language'].unique():
    # Filter the DataFrame for the current language
    lang_df = crs_test[crs_test['language'] == lang]
    
    # Process the batch and update the original DataFrame
    crs_test.loc[lang_df.index, 'lemmatized_title'] = lemmatize_batch(lang_df['normalized_title'].tolist(), lang, batch_size=1000, remove_stopwords=False)

In [ ]:
# Lowercase lemmatized title
crs_test['lemmatized_title'] = crs_test['lemmatized_title'].str.lower()

# Detect keywords in the lemmatized title
crs_test['stat_keywords'] = crs_test.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_keywords), axis=1)
crs_test['gen_keywords'] = crs_test.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], gender_keywords), axis=1)
crs_test['stat_acronyms'] = crs_test.apply(lambda row: detect_keywords(row['lemmatized_title'], row['language'], stat_acronyms), axis=1)

# Detect acronyms in the lemmatized title
crs_test['stat_acronyms'] = crs_test.apply(lambda row: detect_acronyms(row['lemmatized_title'], row['language'], stat_acronyms), axis=1)
crs_test['gen_acronyms'] = crs_test.apply(lambda row: detect_acronyms(row['lemmatized_title'], row['language'], gender_acronyms), axis=1)

In [ ]:
# Save result
crs_test.to_feather("../../data/processed/crs_title_stat_matched.feather")